In [98]:
import numpy as np
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr, pearsonr

# ----------------------------
# Load GloVe embeddings
# ----------------------------
def load_glove_embeddings(glove_file):
    embeddings = {}
    with open(glove_file, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)
            embeddings[word] = vector
    return embeddings

# ----------------------------
# Load CARD-660 pairs
# Format: word1 word2 similarity_score
# ----------------------------
def load_card660_dataset(card_file):
    pairs = []
    with open(card_file, encoding="utf-8") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.strip().split("\t")
                # expected: w1, w2, score
                w1, w2, score = parts[0], parts[1], float(parts[2])
                pairs.append((w1.lower(), w2.lower(), score))
    return pairs

# ----------------------------
# Compute cosine similarity
# ----------------------------
def cosine_similarity(vec1, vec2):
    # 1 - cosine_distance
    return 1 - cosine(vec1, vec2)

# ----------------------------
# Evaluate model
# ----------------------------
def evaluate_glove(glove_embeddings, pairs):
    gold_scores = []
    model_scores = []
    missing = 0

    for w1, w2, gold in pairs:
        if w1 in glove_embeddings and w2 in glove_embeddings:
            vec1 = glove_embeddings[w1]
            vec2 = glove_embeddings[w2]
            sim = cosine_similarity(vec1, vec2)
            gold_scores.append(gold)
            model_scores.append(sim)
        else:
            missing += 1

    print(f"Coverage: {len(gold_scores)} / {len(pairs)} pairs (missing {missing})")
    if len(model_scores) == 0:
        print("No overlap between dataset and embeddings!")
        return None

    # Pearson & Spearman correlations
    pearson_r, _ = pearsonr(model_scores, gold_scores)
    spearman_rho, _ = spearmanr(model_scores, gold_scores)
    print(f"Pearson r: {pearson_r:.4f}")
    print(f"Spearman ρ: {spearman_rho:.4f}")

    return pearson_r, spearman_rho

# ----------------------------
# Run it
# ----------------------------
glove_file = "glove/glove.6B.100d.txt"       
card_file = "benchmark/card-660/dataset.tsv" 

print("=== Base GloVe Evaluation Summary ===")
glove_embeddings = load_glove_embeddings(glove_file)
card_pairs = load_card660_dataset(card_file)

evaluate_glove(glove_embeddings, card_pairs)


=== Base GloVe Evaluation Summary ===
Coverage: 199 / 660 pairs (missing 461)
Pearson r: 0.4487
Spearman ρ: 0.4723


(0.44868023305530225, 0.4723264127879204)

In [81]:
def find_roots(word, glove_embeddings, min_len=5, top_k=2, allow_overlap=False):
    """
    Find top_k longest root substrings.
    If allow_overlap=False, will not allow overlapping roots.
    If allow_overlap=True, allows overlaps.
    """
    candidates = []
    for i in range(len(word)):
        for j in range(i + min_len, len(word) + 1):
            sub = word[i:j]
            if sub in glove_embeddings:
                candidates.append((sub, i, j))

    # Sort candidates by length descending
    candidates.sort(key=lambda x: len(x[0]), reverse=True)

    selected = []
    occupied = set()
    for sub, start, end in candidates:
        if allow_overlap or not any(idx in occupied for idx in range(start, end)):
            selected.append(sub)
            occupied.update(range(start, end))
        if len(selected) >= top_k:
            break

    return selected


def estimate_embedding_filtered(word, glove_embeddings,
                                min_len=5, top_k=2,
                                min_roots_required=1):
    """
    Estimate OOV embedding by aggregating embeddings of top_k
    longest root words:
    - First try non-overlapping roots.
    - If fewer than min_roots_required, allow overlaps as fallback.
    """
    # Exact match
    if word in glove_embeddings:
        return glove_embeddings[word]

    # First: try non-overlapping
    roots = find_roots(word, glove_embeddings, min_len=min_len, top_k=top_k, allow_overlap=False)

    # Fallback: allow overlaps if we didn't get enough roots
    if len(roots) < min_roots_required:
        roots = find_roots(word, glove_embeddings, min_len=min_len, top_k=top_k, allow_overlap=True)

    # Still not enough roots? return None
    # if len(roots) < min_roots_required:
    #     return None
    if len(roots) < min_roots_required:
        # if len(roots) == 1:
        #     return glove_embeddings[roots[0]]

        if len(roots) < 1:
            return None
        
        else:
            return glove_embeddings[roots[0]]

    vecs = np.array([glove_embeddings[r] for r in roots])
    weights = np.array([len(r) for r in roots], dtype=np.float32)
    weights /= np.sum(weights)  # normalize
    return np.sum(vecs * weights[:, np.newaxis], axis=0)


In [96]:
def evaluate_glove_with_filtered_roots(glove_embeddings, pairs,
                                       min_len=5, top_k=2, min_roots_required=1):
    gold_scores = []
    model_scores = []

    # Counters
    exact_count = 0
    estimated_count = 0
    missing_word_count = 0
    missing_pair_count = 0

    total_words = len(pairs) * 2  # two words per pair

    for w1, w2, gold in pairs:
        vecs = []
        for w in [w1, w2]:
            if w in glove_embeddings:
                vecs.append(glove_embeddings[w])
                exact_count += 1
            else:
                vec = estimate_embedding_filtered(
                    w, glove_embeddings,
                    min_len=min_len, top_k=top_k,
                    min_roots_required=min_roots_required
                )
                if vec is not None:
                    vecs.append(vec)
                    estimated_count += 1
                else:
                    vecs.append(None)
                    missing_word_count += 1

        vec1, vec2 = vecs
        if vec1 is None or vec2 is None:
            missing_pair_count += 1
            continue  # skip this pair

        sim = cosine_similarity(vec1, vec2)
        gold_scores.append(gold)
        model_scores.append(sim)

    if not model_scores:
        print("No overlap between dataset and embeddings!")
        return None

    pearson_r, _ = pearsonr(model_scores, gold_scores)
    spearman_rho, _ = spearmanr(model_scores, gold_scores)

    # Percentages
    total_pairs = len(pairs)
    missing_word_pct = 100 * missing_word_count / total_words
    missing_pair_pct = 100 * missing_pair_count / total_pairs

    print("=== Filtered Evaluation Summary ===")
    print(f"Pairs evaluated: {len(gold_scores)} / {total_pairs} "
          f"({100*len(gold_scores)/total_pairs:.2f}%)")
    print(f"Words using exact GloVe vector: {exact_count}")
    print(f"Words using estimated root vector: {estimated_count}")
    print(f"Words missing completely: {missing_word_count} "
          f"({missing_word_pct:.2f}%)")
    print(f"Pairs with missing words skipped: {missing_pair_count} "
          f"({missing_pair_pct:.2f}%)")
    print(f"Pearson r: {pearson_r:.4f}")
    print(f"Spearman ρ: {spearman_rho:.4f}")

    return pearson_r, spearman_rho

# ----------------------------
# Run it
# ----------------------------

glove_embeddings = load_glove_embeddings(glove_file)
card_pairs = load_card660_dataset(card_file)

print("=== GloVe Evaluation with Lemmatization/OOV Handling ===")
evaluate_glove_with_filtered_roots(
    glove_embeddings,
    card_pairs,         
    min_len=6,          
    top_k=3,            
    min_roots_required=3  
)

=== GloVe Evaluation with Lemmatization/OOV Handling ===
=== Filtered Evaluation Summary ===
Pairs evaluated: 422 / 660 (63.94%)
Words using exact GloVe vector: 707
Words using estimated root vector: 342
Words missing completely: 271 (20.53%)
Pairs with missing words skipped: 238 (36.06%)
Pearson r: 0.3176
Spearman ρ: 0.3455


(0.31759494827340734, 0.34550840710874714)